In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (740 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 122797 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
!ollama --version

In [4]:
import subprocess
import time

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

time.sleep(5)

print("Ollama server started successfully")

Ollama server started successfully


In [5]:
!ollama pull llama3.2:3b

In [6]:
!ollama list

NAME           ID              SIZE      MODIFIED               
llama3.2:3b    a80c4f17acd5    2.0 GB    Less than a second ago    


In [7]:
!pip install -q langchain langchain-core langchain-ollama

In [8]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import OllamaLLM
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

print("LangChain imported successfully")

LangChain imported successfully


In [9]:
llm = OllamaLLM(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)

print("Ollama LLM ready")

Ollama LLM ready


In [10]:
prompt = PromptTemplate(
    input_variables=["topic"],
    template="""
Explain {topic} in simple terms for an engineering student.
Give only 2-3 short sentences.
"""
)

parser = StrOutputParser()

chain = prompt | llm | parser

print("PromptTemplate → Ollama LLM → OutputParser")

PromptTemplate → Ollama LLM → OutputParser


In [11]:
topics = [
    "Artificial Intelligence",
    "Machine Learning",
    "Deep Learning",
    "Generative AI",
    "Large Language Models"
]

for i, topic in enumerate(topics, 1):
    result = chain.invoke({"topic": topic})

    print(f"\nTest {i}: {topic}")
    print(result)
    print("-" * 40)


Test 1: Artificial Intelligence
As an engineering student, you know that Artificial Intelligence (AI) refers to the development of computer systems that can perform tasks that typically require human intelligence, such as learning, problem-solving, and decision-making. AI systems use algorithms and data to make predictions, classify objects, and generate insights, making them useful for applications like computer vision, natural language processing, and robotics. In essence, AI enables machines to "think" and act like humans, but with computational power and speed.
----------------------------------------

Test 2: Machine Learning
Here's a simple explanation of Machine Learning for an engineering student:

Machine Learning (ML) is a subset of Artificial Intelligence that enables computers to learn from data without being explicitly programmed. It uses algorithms to analyze and improve performance on a task by identifying patterns and making predictions or decisions based on that data.

In [12]:
conversation_history = []

print("Conversation memory initialized")

Conversation memory initialized


In [13]:
memory_prompt = PromptTemplate(
    input_variables=["history", "input"],
    template="""
You are a helpful assistant.

Previous conversation:
{history}

Current user message:
{input}

Use the previous conversation when needed.
Keep your answer to 2-3 short sentences.
"""
)

memory_chain = memory_prompt | llm | parser

print("Memory chain created")

Memory chain created


In [14]:
def chat_with_memory(user_input):

    history_text = "\n".join(
        [
            f"User: {msg.content}" if isinstance(msg, HumanMessage)
            else f"AI: {msg.content}"
            for msg in conversation_history
        ]
    )

    response = memory_chain.invoke({
        "history": history_text,
        "input": user_input
    })

    conversation_history.append(
        HumanMessage(content=user_input)
    )

    conversation_history.append(
        AIMessage(content=response)
    )

    return response

In [15]:
turns = [
    "My name is Bhavana.",
    "I am an engineering student.",
    "What am I studying?",
    "What is my name?",
    "Suggest a simple AI project for me."
]

for i, message in enumerate(turns, 1):

    response = chat_with_memory(message)

    print(f"\nTurn {i}")
    print("User:", message)
    print("AI:", response)
    print("-" * 40)


Turn 1
User: My name is Bhavana.
AI: Hello Bhavana! Unfortunately, this is the start of our conversation, so I don't have any previous information to draw from. How can I assist you today?
----------------------------------------

Turn 2
User: I am an engineering student.
AI: Hello Bhavana! As an engineering student, you're likely facing various challenges in your academic journey. I'm here to help you with any technical, study-related, or general questions you may have. How can I assist you today?
----------------------------------------

Turn 3
User: What am I studying?
AI: Bhavana, as an engineering student, you're likely pursuing a degree in a field such as mechanical, electrical, civil, or computer science engineering. I can help you with information about various engineering disciplines, course materials, study tips, and more. What specific area of engineering are you interested in learning more about?
----------------------------------------

Turn 4
User: What is my name?
AI: B

In [16]:
print("===== CONVERSATION HISTORY =====\n")

for message in conversation_history:

    if isinstance(message, HumanMessage):
        print("User:", message.content)
    else:
        print("AI:", message.content)

    print()

===== CONVERSATION HISTORY =====

User: My name is Bhavana.

AI: Hello Bhavana! Unfortunately, this is the start of our conversation, so I don't have any previous information to draw from. How can I assist you today?

User: I am an engineering student.

AI: Hello Bhavana! As an engineering student, you're likely facing various challenges in your academic journey. I'm here to help you with any technical, study-related, or general questions you may have. How can I assist you today?

User: What am I studying?

AI: Bhavana, as an engineering student, you're likely pursuing a degree in a field such as mechanical, electrical, civil, or computer science engineering. I can help you with information about various engineering disciplines, course materials, study tips, and more. What specific area of engineering are you interested in learning more about?

User: What is my name?

AI: Bhavana, you've already told me your name earlier in our conversation. You're an engineering student, and we discus

In [17]:
from langchain_core.tools import tool

@tool
def web_search(query: str) -> str:
    """Simulated web search tool."""
    return f"Search result for '{query}': This is a simulated web search result."


@tool
def calculator(expression: str) -> str:
    """Calculator tool."""
    try:
        return str(eval(expression, {"__builtins__": {}}))
    except Exception as e:
        return f"Calculation error: {e}"


print("Tools created successfully")
print("1. Web Search")
print("2. Calculator")

Tools created successfully
1. Web Search
2. Calculator


In [18]:
print("Web Search:")
print(web_search.invoke("Generative AI applications"))

print("\nCalculator:")
print(calculator.invoke("25 * 8"))

Web Search:
Search result for 'Generative AI applications': This is a simulated web search result.

Calculator:
200


In [19]:
def simple_agent(task):

    task_lower = task.lower()

    if any(symbol in task_lower for symbol in [
        "calculate", "*", "/", "+", "-"
    ]):

        expression = task_lower.replace("calculate", "").strip()

        result = calculator.invoke(expression)

        return f"Calculator result: {result}"

    else:

        result = web_search.invoke(task)

        return f"Search result: {result}"

In [20]:
tasks = [
    "Calculate 25 * 8",
    "Calculate 500 / 20",
    "Search for Generative AI applications"
]

for i, task in enumerate(tasks, 1):

    print(f"\nAgent Task {i}")
    print("Input:", task)
    print("Output:", simple_agent(task))
    print("-" * 40)


Agent Task 1
Input: Calculate 25 * 8
Output: Calculator result: 200
----------------------------------------

Agent Task 2
Input: Calculate 500 / 20
Output: Calculator result: 25.0
----------------------------------------

Agent Task 3
Input: Search for Generative AI applications
Output: Search result: Search result for 'Search for Generative AI applications': This is a simulated web search result.
----------------------------------------
